# Packages

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import sys
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.display import clear_output
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


sys.path.insert(0, "Module")
import Dataset_exoplanète as utils
from Processing import Processing
from Training import train_and_evaluate
from output_width import get_output_width
from EarlyStopper import EarlyStopper

# Model

In [ ]:
class ExoplanetNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv1d(1,16,kernel_size=7,padding='same'),
            nn.BatchNorm1d(16),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(16,32,kernel_size=5,padding='same'),
            nn.BatchNorm1d(32),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm1d(64),
            nn.ELU(),
        )
        self.linear_entry1 = get_output_width(3197, kernel_size=7, padding=3)
        self.linear_entry2 = get_output_width(self.linear_entry1, kernel_size=2, stride=2)
        self.linear_entry3 = get_output_width(self.linear_entry2, kernel_size=5, padding=2)
        self.linear_entry4 = get_output_width(self.linear_entry3, kernel_size=2, stride=2)
        self.linear_entry = get_output_width(self.linear_entry4, kernel_size=3, padding=1)

        self.flatten = nn.Flatten()

        self.linear_stack = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(64*self.linear_entry, 128),
            nn.Dropout(0.2),
            nn.ELU(),
            nn.Linear(128, 128),
            nn.Dropout(0.2),
            nn.ELU(),
            nn.Linear(128, 64),
            nn.ELU(),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Linear(32, 1),

        )
    def forward(self, x):
        x = self.conv_stack(x)
        x = self.flatten(x)
        x = self.linear_stack(x)
        return x

model = ExoplanetNN().to(device)


# Training

## Creation of the Dataset and Dataloader

In [ ]:
train_dataset = utils.ExoplanetDataset('Train/exoTrain')
test_dataset = utils.ExoplanetDataset('Train/exoTest')

print("SANS OVERSAMPLING")
print("Données d'entraînement:")
print("----------------------------------------")
print("Nombre de courbe avec le label 0:",len(np.where(train_dataset.labels == 0)[0]))
print("Nombre de courbe avec le label 1:",len(np.where(train_dataset.labels == 1)[0]))
print("----------------------------------------")
print()
print("Données de test:")
print("----------------------------------------")
print("Nombre de courbe avec le label 0:",len(np.where(test_dataset.labels == 0)[0]))
print("Nombre de courbe avec le label 1:",len(np.where(test_dataset.labels == 1)[0]))
print("----------------------------------------")
print()
print()

batch_size = 64

train_dataset = Processing(train_dataset, num_iterations=3, do_oversampling=True)
test_dataset = Processing(test_dataset, num_iterations=3, do_oversampling=False)


train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False, num_workers=4, pin_memory=True)
print("AVEC OVERSAMPLING")
print("Données d'entraînement:")
print("----------------------------------------")
print("Nombre de courbe avec le label 0:",len(np.where(train_dataset.tensors[1] == 0)[0]))
print("Nombre de courbe avec le label 1:",len(np.where(train_dataset.tensors[1] == 1)[0]))
print("----------------------------------------")
print()
print("Données de test:")
print("----------------------------------------")
print("Nombre de courbe avec le label 0:",len(np.where(test_dataset.tensors[1] == 0)[0]))
print("Nombre de courbe avec le label 1:",len(np.where(test_dataset.tensors[1] == 1)[0]))
print("----------------------------------------")

In [ ]:
# Vérification des dimensions des batchs
X_batch, y_batch = next(iter(train_dataloader))

print("Dimensions de X_batch :", X_batch.shape)
print("Dimensions de y_batch :", y_batch.shape)

### Affichage de courbe avec et sans exoplanète

In [ ]:
idx_exo = np.where(test_dataset.tensors[1] == 1)[0]
idx_exo = np.random.choice(idx_exo)

idx_non_exo = np.where(test_dataset.tensors[1] == 0)[0]
idx_non_exo = np.random.choice(idx_non_exo)

print(f"Exoplanète : {idx_exo}, Non-exoplanète : {idx_non_exo}")

plt.figure(figsize=(15,3))
plt.plot(test_dataset.tensors[0][idx_exo].squeeze())
plt.title("Présence d'exoplanète(s)")
plt.xlabel("Temps")
plt.ylabel("Flux ($e^-/s$)")

plt.figure(figsize=(15,3))
plt.plot(test_dataset.tensors[0][idx_non_exo].squeeze())
plt.title("Sans exoplanète")
plt.xlabel("Temps")
plt.ylabel("Flux ($e^-/s$)")
plt.show()

## Training parameters

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([15], device=device))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)
epochs=50
early_stopper = EarlyStopper(patience=8, min_delta=0)

## Training on 1 model

In [ ]:
threshold = 0.50
train_losses, val_losses, cm, best_val_loss, cm_f1, best_f1_score, model_val, model_f1 = train_and_evaluate(train_dataloader, test_dataloader, model, loss_fn, optimizer, epochs, device, scheduler, early_stopper, threshold)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[utils.CDICT[0], utils.CDICT[1]])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matrice de Confusion du Meilleur Modèle Eval Loss")
plt.show()
print(f"Exactitude: {(cm[0,0]+cm[1,1])/cm.sum():.4f}")
print(f"Précision: {cm[1,1]/(cm[0,1]+cm[1,1]):.4f}")
print(f"Rappel: {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}")
print(f"F1-Score: {2*cm[1,1]/(2*cm[1,1]+cm[0,1]+cm[1,0]):.4f}")
print()


disp_f1 = ConfusionMatrixDisplay(confusion_matrix=cm_f1, display_labels=[utils.CDICT[0], utils.CDICT[1]])
disp_f1.plot(cmap=plt.cm.Oranges)
plt.title("Matrice de Confusion du Meilleur Modèle F1-Score")
plt.show()
print(f"Exactitude: {(cm_f1[0,0]+cm_f1[1,1])/cm_f1.sum():.4f}")
print(f"Précision: {cm_f1[1,1]/(cm_f1[0,1]+cm_f1[1,1]):.4f}")
print(f"Rappel: {cm_f1[1,1]/(cm_f1[1,0]+cm_f1[1,1]):.4f}")
print(f"F1-Score: {2*cm_f1[1,1]/(2*cm_f1[1,1]+cm_f1[0,1]+cm_f1[1,0]):.4f}")
print()


torch.save({
    'model_state_dict': model_val.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'best_val_loss': best_val_loss,
    'confusion_matrix': cm.tolist(),
}, r'Modèles\modèle_optimisé_val.pth')

torch.save({
    'model_state_dict': model_f1.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'best_f1_score': best_f1_score,
    'confusion_matrix': cm_f1.tolist(),
}, r'Modèles\modèle_optimisé_f1.pth')

## Training on several models

In [ ]:
def metrics_from_cm(cm):
    tn, fp = cm[0, 0], cm[0, 1]
    fn, tp = cm[1, 0], cm[1, 1]

    accuracy = (tn + tp) / cm.sum() if cm.sum() > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0

    return accuracy, precision, recall, f1


n_runs = 20

acc_list = []
prec_list = []
rec_list = []
f1_list = []
val_loss_list = []

all_results = []

for run in range(n_runs):
    print(f"\n===== Entraînement {run+1}/{n_runs} =====")

    model = ExoplanetNN().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )

    early_stopper = EarlyStopper(patience=8, min_delta=0)

    train_losses, val_losses, cm, best_val_loss, cm_f1, best_f1_score, model_val, model_f1 = train_and_evaluate(
        train_dataloader,
        test_dataloader,
        model,
        loss_fn,
        optimizer,
        epochs,
        device,
        scheduler,
        early_stopper,
        threshold
    )

    accuracy, precision, recall, f1 = metrics_from_cm(cm_f1)

    acc_list.append(accuracy)
    prec_list.append(precision)
    rec_list.append(recall)
    f1_list.append(f1)
    val_loss_list.append(best_val_loss)

    all_results.append({
        "run": run + 1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "best_val_loss": best_val_loss,
        "cm_f1": cm_f1,
        "cm_val": cm
    })
clear_output(wait=True)

### Graphs

In [ ]:
metrics_dict = {
    "Exactitude": acc_list,
    "Précision": prec_list,
    "Rappel": rec_list,
    "F1-score": f1_list,
}
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
axes = axes.flatten()

for i, (nom, valeurs) in enumerate(metrics_dict.items()):
    moyenne = np.mean(valeurs)
    ecart_type = np.std(valeurs)

    axes[i].hist(valeurs, bins=30, edgecolor='black')
    axes[i].axvline(
        moyenne,
        linestyle='--',
        linewidth=2,
        label=f"Moy = {moyenne:.4f}\nStd = {ecart_type:.4f}"
    )
    axes[i].set_title(f"Distribution de {nom}")
    axes[i].set_xlabel(nom)
    axes[i].set_ylabel("Nombre d'itérations")
    axes[i].legend()

if len(metrics_dict) < len(axes):
    for j in range(len(metrics_dict), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()